# First look: ดูโครงสร้าง ช่วงเวลา หน่วย และค่าหาย ไม่วิเคราะห์

การสำรวจข้อมูลเบื้องต้น (First look) มีวัตถุประสงค์เพื่อตรวจสอบว่าข้อมูลมีโครงสร้าง ช่วงเวลา หน่วยวัด และสัดส่วนค่าหายเป็นอย่างไร เพียงพอที่จะตอบคำถามเชิงวิเคราะห์ใน `questions.md` หรือไม่ โดยยังไม่ทำการเปรียบเทียบกลุ่มหรือสรุปผลเกินหลักฐาน

In [1]:
import requests
import pandas as pd
import truststore; truststore.inject_into_ssl()     # ดูตารางแก้ปัญหา: SSLError

URL = "http://air4thai.pcd.go.th/services/getNewAQI_JSON.php"
r = requests.get(URL, timeout=30)
r.status_code, r.headers.get("Content-Type")

(200, 'application/json; charset=UTF-8')

In [2]:
data = r.json()
print(type(data), list(data.keys()) if isinstance(data, dict) else len(data))
stations = data["stations"]          # ถ้า key ไม่ชื่อนี้ ให้แก้ตามที่เห็นจาก cell ก่อนหน้า
print(len(stations))
stations[0]                          # ดูโครงสร้างของสถานีแรกทั้งก้อน

<class 'dict'> ['stations']
178


{'stationID': '119t',
 'nameTH': 'สวนสาธารณะธารา',
 'nameEN': 'Thara Public Park',
 'areaTH': 'ต.ปากน้ำ อ.เมือง, กระบี่',
 'areaEN': 'Pak Nam Subdistrict, Mueang District, Krabi',
 'stationType': 'GROUND',
 'lat': '8.0506237',
 'long': '98.9180489',
 'forecast': [],
 'AQILast': {'date': '2026-11-18',
  'time': '21:00',
  'PM25': {'color_id': '1', 'aqi': '10', 'value': '5.9'},
  'PM10': {'color_id': '0', 'aqi': '-1', 'value': '-1'},
  'O3': {'color_id': '1', 'aqi': '16', 'value': '22'},
  'CO': {'color_id': '1', 'aqi': '2', 'value': '0.30'},
  'NO2': {'color_id': '1', 'aqi': '0', 'value': '1'},
  'SO2': {'color_id': '1', 'aqi': '0', 'value': '1'},
  'AQI': {'color_id': '1', 'aqi': '16', 'param': 'O3'}}}

In [3]:
df_now = pd.json_normalize(stations)
print(df_now.shape)
df_now.columns.tolist()              # ชื่อคอลัมน์ที่ซ้อนกันจะต่อด้วยจุด เช่น "<key>.<subkey>"

(178, 32)


['stationID',
 'nameTH',
 'nameEN',
 'areaTH',
 'areaEN',
 'stationType',
 'lat',
 'long',
 'forecast',
 'AQILast.date',
 'AQILast.time',
 'AQILast.PM25.color_id',
 'AQILast.PM25.aqi',
 'AQILast.PM25.value',
 'AQILast.PM10.color_id',
 'AQILast.PM10.aqi',
 'AQILast.PM10.value',
 'AQILast.O3.color_id',
 'AQILast.O3.aqi',
 'AQILast.O3.value',
 'AQILast.CO.color_id',
 'AQILast.CO.aqi',
 'AQILast.CO.value',
 'AQILast.NO2.color_id',
 'AQILast.NO2.aqi',
 'AQILast.NO2.value',
 'AQILast.SO2.color_id',
 'AQILast.SO2.aqi',
 'AQILast.SO2.value',
 'AQILast.AQI.color_id',
 'AQILast.AQI.aqi',
 'AQILast.AQI.param']

### สรุปสิ่งที่เห็นจากข้อมูลปัจจุบันของ Air4Thai (ขั้นที่ 4)

1. **หนึ่งแถวของ `df_now` คืออะไร:** หนึ่งแถวแทนข้อมูลของ **1 สถานีตรวจวัดคุณภาพอากาศ ณ เวลาที่รายงานล่าสุด** (`AQILast`)
2. **คอลัมน์ที่เก็บรหัสสถานี ชื่อ พิกัด วันที่ และเวลา:**
   - รหัสสถานี: `stationID`
   - ชื่อสถานี: `nameTH` (ภาษาไทย) และ `nameEN` (ภาษาอังกฤษ)
   - ที่ตั้ง/พื้นที่: `areaTH` และ `areaEN` (ต้องสกัดชื่อจังหวัดออกจากข้อความนี้ และยังไม่มีคอลัมน์ภาค `region` มาให้โดยตรง)
   - พิกัดภูมิศาสตร์: `lat` (ละติจูด) และ `long` (ลองจิจูด)
   - วันที่และเวลาล่าสุด: `AQILast.date` และ `AQILast.time`
3. **ค่า PM2.5 อยู่คอลัมน์ไหน มีทั้งค่าความเข้มข้นและค่า AQI หรือไม่ หน่วยคืออะไร:**
   - มีแยกกันทั้ง 2 ค่า คือ **`AQILast.PM25.value`** (ค่าความเข้มข้นของฝุ่น PM2.5 หน่วยเป็นไมโครกรัมต่อลูกบาศก์เมตร หรือ **µg/m³**) และ **`AQILast.PM25.aqi`** (ค่าดัชนีคุณภาพอากาศ AQI ซึ่งเป็นตัวเลขไร้หน่วยพร้อมรหัสสีใน `AQILast.PM25.color_id`)
4. **ชนิดข้อมูลและค่าแปลกแทนค่าหาย:**
   - ค่าตัวเลขทั้งหมดใน JSON ของ Air4Thai (ทั้ง `lat`, `long`, `AQILast.PM25.value`, `AQILast.PM25.aqi`) ถูกส่งมาเป็นข้อความ (`object` / `string`)
   - เมื่อแปลง `AQILast.PM25.value` เป็นตัวเลขด้วย `pd.to_numeric(..., errors="coerce")` ใน cell ถัดไป พบว่ามี **ค่าติดลบ (`-1` หรือ `-1.0`)** ซึ่ง Air4Thai ใช้เป็นรหัสแทนข้อมูลสูญหายหรือเครื่องมือไม่ได้วัดค่าในชั่วโมงนั้น จึงต้องแปลงค่าติดลบเป็น `NaN` / `NULL` ก่อนนำไปคำนวณเสมอ มิฉะนั้นค่าเฉลี่ยจะต่ำกว่าความเป็นจริง

In [4]:
pm = pd.to_numeric(df_now["AQILast.PM25.value"], errors="coerce")
print("แปลงเป็นตัวเลขไม่ได้:", pm.isna().sum())
print("ค่าติดลบ:", (pm < 0).sum())
pm.describe()

แปลงเป็นตัวเลขไม่ได้: 0
ค่าติดลบ: 2


count    178.000000
mean      18.563483
std        6.465236
min       -1.000000
25%       15.425000
50%       18.900000
75%       22.300000
max       36.700000
Name: AQILast.PM25.value, dtype: float64

### ขั้นที่ 5: First look ไฟล์ตัวอย่างย้อนหลัง (เฉพาะช่วงสำรวจ 2563–2566)

เนื่องจากข้อมูลจาก API ปัจจุบันมีเพียงจุดเวลาล่าสุดของแต่ละสถานี จึงยังไม่สามารถตรวจสอบความถี่รายชั่วโมงและสัดส่วนค่าหายตามช่วงเวลาได้ ในส่วนนี้จึงตรวจสอบโครงสร้างและค่าหายจากไฟล์ตัวอย่างย้อนหลัง โดยจำกัดเฉพาะ **ช่วงสำรวจ (`ts < "2024-01-01"`)** เท่านั้น และไม่เปิดดูข้อมูลช่วงทดสอบ (`2024–2025`)

In [5]:
hourly = pd.read_csv("../data/sample/pm25_hourly_sample.csv", parse_dates=["ts"])
st = pd.read_csv("../data/sample/stations_sample.csv")

# ใช้เฉพาะช่วงสำรวจ 2563–2566 (ค.ศ. 2020–2023) ห้ามดูช่วงทดสอบ
explore = hourly[hourly["ts"] < "2024-01-01"]

print(explore.shape, explore["station_id"].nunique(), "สถานี")
print(explore["ts"].min(), "→", explore["ts"].max())
print(explore["ts"].diff().value_counts().head(3))   # ความถี่ของข้อมูล (ควรเป็น 1 ชั่วโมง)
explore.isna().mean().round(3)

(131400, 8) 15 สถานี
2023-01-01 00:00:00 → 2023-12-31 23:00:00
ts
0 days 01:00:00        131385
-365 days +01:00:00        14
Name: count, dtype: int64


station_id    0.000
ts            0.000
pm25          0.085
pm10          0.406
o3            0.781
co            0.801
no2           0.792
so2           0.821
dtype: float64

In [6]:
# สถานีที่ค่า PM2.5 หายมากที่สุด 10 อันดับ (ดูว่าค่าหายอยู่ตรงไหน ยังไม่ตัดสินว่าทำไม)
(explore.groupby("station_id")["pm25"]
        .apply(lambda s: s.isna().mean())
        .sort_values(ascending=False)
        .head(10))

station_id
119t       0.267922
02t        0.183333
bkp103t    0.080822
12t        0.080251
59t        0.080023
05t        0.073516
bkp102t    0.071005
54t        0.070320
03t        0.062900
53t        0.059589
Name: pm25, dtype: float64

### สรุปสิ่งที่พบจาก First look (ขั้นที่ 4–5)

1. **รูปแบบการแทนค่าหายใน API ปัจจุบัน:** ข้อมูลจาก `getNewAQI_JSON.php` เก็บค่าตัวเลขทั้งหมดเป็นข้อความ (`string`) และพบว่ามีสถานีที่ส่งค่า `AQILast.PM25.value` เป็น **`-1`** แทนข้อมูลที่ขาดหาย ซึ่งควรตรวจสอบต่อและกำหนดกฎการทำความสะอาดข้อมูลให้แปลงค่า `< 0` เป็น `NULL` ก่อนคำนวณทุกครั้ง
2. **ความถี่และขอบเขตของข้อมูลช่วงสำรวจ:** ข้อมูลย้อนหลังในช่วงสำรวจ (`ts < "2024-01-01"`) มีความถี่ปกติเป็นราย 1 ชั่วโมง (`0 days 01:00:00`) ครอบคลุม 15 สถานีในชุดตัวอย่าง โดยไม่มีแถวของช่วงทดสอบ (`>= 2024-01-01`) ปะปนเข้ามา
3. **สัดส่วนค่าหายระหว่างสารมลพิษแต่ละชนิด:** ข้อมูลในช่วงสำรวจบ่งชี้ว่าคอลัมน์ `pm25` มีสัดส่วนค่าหายโดยรวมประมาณร้อยละ 8.5 (`0.085`) ขณะที่ `pm10` หายประมาณร้อยละ 40.6 (`0.406`) และก๊าซอื่น ๆ (`o3`, `co`, `no2`, `so2`) มีค่าหายสูงถึงร้อยละ 78.1–82.1 จึงควรตรวจสอบต่อว่าเกิดจากการที่บางสถานีไม่ได้ติดตั้งเครื่องวัดก๊าซเหล่านั้นตั้งแต่ต้นหรือไม่
4. **ความไม่สม่ำเสมอของค่าหายรายสถานี:** เมื่อจำแนกตามรหัสสถานี พบแนวโน้มว่าบางสถานีมีสัดส่วนค่า `pm25` หายสูงกว่าสถานีอื่นอย่างชัดเจน (เช่น สถานี `119t` มีค่าหายร้อยละ 26.8 และสถานี `02t` หายร้อยละ 18.3 ในขณะที่สถานีส่วนใหญ่หายเพียงร้อยละ 6–8) จึงควรตั้งคำถาม data-driven เพื่อตรวจสอบต่อว่าค่าหายเหล่านี้กระจุกตัวอยู่ในเดือนฤดูฝุ่น (เช่น มี.ค.–เม.ย.) หรือไม่ เพราะอาจส่งผลต่อการคำนวณสัดส่วนวันเกินมาตรฐานใน Q1
5. **โครงสร้างข้อมูลที่ตั้งสถานี:** ข้อมูลจาก API ปัจจุบันไม่มีคอลัมน์ `province` และ `region` แยกมาให้โดยตรง แต่รวมอยู่ในข้อความ `areaTH` ดังนั้นในการสร้างฐานข้อมูลสัปดาห์ถัดไป จึงต้องสกัดชื่อจังหวัดและสร้างตารางเชื่อมโยงจังหวัดเข้ากับภูมิภาคเพื่อให้ตอบคำถาม Q1 ได้